In [1]:
!pip install ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
import json
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO
import torchvision.transforms as T
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_DIR    = "/kaggle/input/datasets/kavishgrover/bdd-oia/data"
OUTPUT_DIR   = "/kaggle/working/vit_features"
YOLO_WEIGHTS = "/kaggle/working/yolov8m.pt"
LOG_PATH     = "/kaggle/working/vit_processed.txt"
device       = "cuda" if torch.cuda.is_available() else "cpu"

ACTIONS_JSON = {
    "train": "/kaggle/input/datasets/kavishgrover/bdd-oia/train_25k_images_actions.json",
    "val"  : "/kaggle/input/datasets/kavishgrover/bdd-oia/val_25k_images_actions.json",
    "test" : "/kaggle/input/datasets/kavishgrover/bdd-oia/test_25k_images_actions.json",
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load already processed ────────────────────────────────────────────────────
already_processed = set()
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        already_processed = set(line.strip() for line in f if line.strip())
    print(f"Resuming — {len(already_processed)} already processed ✓")
else:
    print("Fresh start ✓")

log_out = open(LOG_PATH, "a")  # append mode so we don't lose previous entries

# ── Load YOLO ─────────────────────────────────────────────────────────────────
if not os.path.exists(YOLO_WEIGHTS):
    print("Downloading yolov8m.pt...")
    _tmp = YOLO("yolov8m.pt")
    del _tmp
yolo_model = YOLO(YOLO_WEIGHTS)
print("YOLO loaded ✓")

# ── Load DINOv2 ───────────────────────────────────────────────────────────────
print("Loading DINOv2...")
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
dino.eval().to(device)
for param in dino.parameters():
    param.requires_grad = False
print("DINOv2 ViT-B/14 loaded and frozen ✓")

# ── Transform ─────────────────────────────────────────────────────────────────
# DINOv2 expects 224×224, normalized with ImageNet stats
# patch size 14 → 224/14 = 16×16 patch grid → 256 patch tokens
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# ── Get filenames for a split (excluding confuse) ─────────────────────────────
def get_filenames(split):
    with open(ACTIONS_JSON[split]) as f:
        actions_data = json.load(f)
    id_to_filename = {img["id"]: img["file_name"] for img in actions_data["images"]}
    filenames = []
    for ann in actions_data["annotations"]:
        cat = ann["category"]
        if len(cat) == 5 and cat[4] == 1:
            continue
        filenames.append(id_to_filename[ann["id"]])
    return filenames

# ── Feature extraction ────────────────────────────────────────────────────────
def extract_vit_features(image_path, conf_threshold=0.3):
    """
    Returns:
      cls_token   : (768,)        — global scene feature
      patch_tokens: (256, 768)    — spatial patch features (16×16 grid flattened)
      boxes       : (N, 4)        — YOLO boxes in 224×224 pixel space
      used_fallback: bool
    """
    from PIL import Image as PILImage
    import cv2

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"Could not read: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = PILImage.fromarray(img_rgb)
    H, W    = img_rgb.shape[:2]

    # ── YOLO ─────────────────────────────────────────────────────────────────
    results   = yolo_model(image_path, conf=conf_threshold, verbose=False)
    raw_boxes = results[0].boxes.xyxy.cpu().numpy()

    # Scale boxes from original image size to 224×224
    scale_x = 224.0 / W
    scale_y = 224.0 / H

    valid_boxes = []
    for box in raw_boxes:
        x_min, y_min, x_max, y_max = box
        x_min = max(0, x_min * scale_x)
        y_min = max(0, y_min * scale_y)
        x_max = min(224, x_max * scale_x)
        y_max = min(224, y_max * scale_y)
        if x_max <= x_min or y_max <= y_min:
            continue
        valid_boxes.append([x_min, y_min, x_max, y_max])

    used_fallback = False
    if len(valid_boxes) == 0:
        valid_boxes   = [[0, 0, 224, 224]]
        used_fallback = True

    boxes = np.array(valid_boxes, dtype=np.float32)  # (N, 4) in 224×224 space

    # ── DINOv2 forward pass ───────────────────────────────────────────────────
    img_tensor = transform(pil_img).unsqueeze(0).to(device)  # (1, 3, 224, 224)

    with torch.no_grad():
        output = dino.forward_features(img_tensor)

    # DINOv2 returns dict with 'x_norm_clstoken' and 'x_norm_patchtokens'
    cls_token    = output["x_norm_clstoken"].squeeze(0).cpu()   # (768,)
    patch_tokens = output["x_norm_patchtokens"].squeeze(0).cpu() # (256, 768)

    return cls_token, patch_tokens, boxes, used_fallback

# ── Preprocessing loop ────────────────────────────────────────────────────────
def preprocess_split(split):
    filenames = get_filenames(split)
    out_dir   = os.path.join(OUTPUT_DIR, split)
    os.makedirs(out_dir, exist_ok=True)

    already_done = sum(1 for f in filenames if f in already_processed)
    remaining    = len(filenames) - already_done

    print(f"\n── {split} ───────────────────────────────────────")
    print(f"  Total   : {len(filenames)}")
    print(f"  Done    : {already_done}")
    print(f"  Remaining: {remaining}")

    if remaining == 0:
        print(f"  {split} fully complete, skipping.")
        return

    failed    = []
    fallbacks = []
    processed = 0

    for fname in tqdm(filenames, desc=split):
        if fname in already_processed:
            continue

        img_path  = os.path.join(IMAGE_DIR, fname)
        save_path = os.path.join(out_dir, fname.replace(".jpg", ".pt"))

        if not os.path.exists(img_path):
            failed.append(fname)
            continue

        try:
            cls_token, patch_tokens, boxes, used_fallback = extract_vit_features(img_path)

            torch.save({
                "filename"    : fname,
                "cls_token"   : cls_token,     # (768,)
                "patch_tokens": patch_tokens,  # (256, 768)
                "boxes"       : torch.tensor(boxes),  # (N, 4) in 224×224 space
            }, save_path)

            already_processed.add(fname)
            log_out.write(fname + "\n")
            log_out.flush()
            processed += 1

            if used_fallback:
                fallbacks.append(fname)

        except Exception as e:
            print(f"  Failed {fname}: {e}")
            failed.append(fname)

    print(f"\n  {split} done.")
    print(f"  Processed this run : {processed}")
    print(f"  Failed             : {len(failed)}")
    print(f"  Fallbacks          : {len(fallbacks)}")

# ── Run all splits ────────────────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    preprocess_split(split)

log_out.close()

# ── Sanity check ─────────────────────────────────────────────────────────────
print("\n── Sanity check ─────────────────────────────────")
for split in ["train", "val", "test"]:
    split_dir = os.path.join(OUTPUT_DIR, split)
    if not os.path.exists(split_dir):
        print(f"  {split}: directory not found")
        continue
    files = os.listdir(split_dir)
    print(f"  {split}: {len(files)} files saved")

first_file = os.listdir(os.path.join(OUTPUT_DIR, "train"))[0]
sample     = torch.load(os.path.join(OUTPUT_DIR, "train", first_file))
print(f"\n  Sample file: {sample['filename']}")
print(f"  cls_token   : {sample['cls_token'].shape}")    # (768,)
print(f"  patch_tokens: {sample['patch_tokens'].shape}") # (256, 768)
print(f"  boxes       : {sample['boxes'].shape}")        # (N, 4)
print(f"\nAll done. Log: {LOG_PATH}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Fresh start ✓
YOLO loaded ✓
Loading DINOv2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:00<00:00, 415MB/s]


DINOv2 ViT-B/14 loaded and frozen ✓

── train ───────────────────────────────────────
  Total   : 16028
  Done    : 0
  Remaining: 16028


...


  train done.
  Processed this run : 16028
  Failed             : 0
  Fallbacks          : 236

── val ───────────────────────────────────────
  Total   : 2259
  Done    : 0
  Remaining: 2259


val: 100%|██████████| 2259/2259 [02:00<00:00, 18.69it/s]



  val done.
  Processed this run : 2259
  Failed             : 0
  Fallbacks          : 26

── test ───────────────────────────────────────
  Total   : 4548
  Done    : 0
  Remaining: 4548


test: 100%|██████████| 4548/4548 [04:05<00:00, 18.56it/s]


  test done.
  Processed this run : 4548
  Failed             : 0
  Fallbacks          : 48

── Sanity check ─────────────────────────────────
  train: 16028 files saved
  val: 2259 files saved
  test: 4548 files saved

  Sample file: 622e382a-a65d9725.jpg
  cls_token   : torch.Size([768])
  patch_tokens: torch.Size([256, 768])
  boxes       : torch.Size([6, 4])

All done. Log: /kaggle/working/vit_processed.txt
